# Data Cleaning

## Project: Albania Brain Drain Analysis

## Goal
 We clean the World Bank Albania dataset.
The goal is to extract the main indicators needed for the analysis:

- Population total
- Population growth
- Net migration
- GDP current US$
- GDP growth annual %
- GDP per capita
- Unemployment rate
- Remittances
- Foreign direct investment

The cleaned files will be saved inside the `data/processed/` folder.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Root:", ROOT_DIR)
print("Raw:", RAW_DIR)
print("Processed:", PROCESSED_DIR)

Root: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis
Raw: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\data\raw
Processed: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\data\processed


### Load the World Bank dataset

In [2]:
worldbank_path = RAW_DIR / "worldbank" / "API_ALB_DS2_en_csv_v2_3304.csv"

wb_raw = pd.read_csv(worldbank_path, skiprows=4)

print("Rows:", wb_raw.shape[0])
print("Columns:", wb_raw.shape[1])

wb_raw.head()

Rows: 1486
Columns: 71


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Albania,ALB,"Internally displaced persons, new displacement...",VC.IDP.NWDS,NaN,NaN,NaN,NaN,NaN,NaN,...,3500.000000,1.100000e+02,3.300000e+04,NaN,250.000000,320.000000,13.000000,NaN,NaN,NaN
1,Albania,ALB,Transport services (% of commercial service ex...,TX.VAL.TRAN.ZS.WT,NaN,NaN,NaN,NaN,NaN,NaN,...,8.397777,8.554160e+00,7.278729e+00,8.239105e+00,10.175969,10.616492,10.844297,10.769175,NaN,NaN
2,Albania,ALB,"Computer, communications and other services (%...",TX.VAL.OTHR.ZS.WT,NaN,NaN,NaN,NaN,NaN,NaN,...,30.505495,2.985076e+01,2.999540e+01,4.502030e+01,33.747498,28.965182,24.455173,20.952803,NaN,NaN
3,Albania,ALB,"Merchandise exports by the reporting economy, ...",TX.VAL.MRCH.RS.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,0.000000,1.923261e-15,-1.996821e-15,-2.194358e-15,0.000000,0.000000,0.002770,NaN,NaN,NaN
4,Albania,ALB,Merchandise exports to low- and middle-income ...,TX.VAL.MRCH.R3.ZS,NaN,NaN,NaN,NaN,NaN,NaN,...,0.035620,9.103126e-02,5.967191e-02,4.745652e-02,0.076538,0.107650,0.059638,NaN,NaN,NaN


### Select the important indicators

In [3]:
selected_indicators = {
    "SP.POP.TOTL": "population_total",
    "SP.POP.GROW": "population_growth_annual_percent",
    "SM.POP.NETM": "net_migration",
    "NY.GDP.MKTP.CD": "gdp_current_usd",
    "NY.GDP.MKTP.KD.ZG": "gdp_growth_annual_percent",
    "NY.GDP.PCAP.CD": "gdp_per_capita_current_usd",
    "SL.UEM.TOTL.ZS": "unemployment_total_percent",
    "BX.TRF.PWKR.DT.GD.ZS": "remittances_percent_gdp",
    "BX.KLT.DINV.WD.GD.ZS": "fdi_percent_gdp"
}

wb_selected = wb_raw[wb_raw["Indicator Code"].isin(selected_indicators.keys())].copy()

wb_selected[["Country Name", "Indicator Name", "Indicator Code"]]

,Country Name,Indicator Name,Indicator Code
327,Albania,"Foreign direct investment, net inflows (% of GDP)",BX.KLT.DINV.WD.GD.ZS
373,Albania,Population growth (annual %),SP.POP.GROW
409,Albania,GDP growth (annual %),NY.GDP.MKTP.KD.ZG
410,Albania,GDP (current US$),NY.GDP.MKTP.CD
769,Albania,Net migration,SM.POP.NETM
771,Albania,"Unemployment, total (% of total labor force) (...",SL.UEM.TOTL.ZS
919,Albania,"Personal remittances, received (% of GDP)",BX.TRF.PWKR.DT.GD.ZS
965,Albania,"Population, total",SP.POP.TOTL
1270,Albania,GDP per capita (current US$),NY.GDP.PCAP.CD


### Convert from wide format to long format

In [4]:
id_columns = ["Country Name", "Country Code", "Indicator Name", "Indicator Code"]

year_columns = [col for col in wb_selected.columns if col.isdigit()]

wb_long = wb_selected.melt(
    id_vars=id_columns,
    value_vars=year_columns,
    var_name="year",
    value_name="value"
)

wb_long["year"] = wb_long["year"].astype(int)
wb_long["value"] = pd.to_numeric(wb_long["value"], errors="coerce")

wb_long["indicator_short_name"] = wb_long["Indicator Code"].map(selected_indicators)

wb_long = wb_long.rename(columns={
    "Country Name": "country_name",
    "Country Code": "country_code",
    "Indicator Name": "indicator_name",
    "Indicator Code": "indicator_code"
})

wb_long = wb_long[
    [
        "country_name",
        "country_code",
        "indicator_name",
        "indicator_code",
        "indicator_short_name",
        "year",
        "value"
    ]
]

wb_long.head()

,country_name,country_code,indicator_name,indicator_code,indicator_short_name,year,value
0,Albania,ALB,"Foreign direct investment, net inflows (% of GDP)",BX.KLT.DINV.WD.GD.ZS,fdi_percent_gdp,1960,NaN
1,Albania,ALB,Population growth (annual %),SP.POP.GROW,population_growth_annual_percent,1960,NaN
2,Albania,ALB,GDP growth (annual %),NY.GDP.MKTP.KD.ZG,gdp_growth_annual_percent,1960,NaN
3,Albania,ALB,GDP (current US$),NY.GDP.MKTP.CD,gdp_current_usd,1960,NaN
4,Albania,ALB,Net migration,SM.POP.NETM,net_migration,1960,13734.0


### Filter years from 2000 onward

In [5]:
wb_clean = wb_long[wb_long["year"] >= 2000].copy()

wb_clean = wb_clean.dropna(subset=["value"])

wb_clean = wb_clean.sort_values(["indicator_short_name", "year"])

print("Rows after cleaning:", wb_clean.shape[0])
print("Year range:", wb_clean["year"].min(), "-", wb_clean["year"].max())

wb_clean.head()

Rows after cleaning: 227
Year range: 2000 - 2025


,country_name,country_code,indicator_name,indicator_code,indicator_short_name,year,value
360,Albania,ALB,"Foreign direct investment, net inflows (% of GDP)",BX.KLT.DINV.WD.GD.ZS,fdi_percent_gdp,2000,3.989321
369,Albania,ALB,"Foreign direct investment, net inflows (% of GDP)",BX.KLT.DINV.WD.GD.ZS,fdi_percent_gdp,2001,5.107089
378,Albania,ALB,"Foreign direct investment, net inflows (% of GDP)",BX.KLT.DINV.WD.GD.ZS,fdi_percent_gdp,2002,2.990031
387,Albania,ALB,"Foreign direct investment, net inflows (% of GDP)",BX.KLT.DINV.WD.GD.ZS,fdi_percent_gdp,2003,3.068687
396,Albania,ALB,"Foreign direct investment, net inflows (% of GDP)",BX.KLT.DINV.WD.GD.ZS,fdi_percent_gdp,2004,4.607823


### Save the main cleaned World Bank file

In [6]:
output_path = PROCESSED_DIR / "worldbank_selected_indicators_clean.csv"

wb_clean.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\data\processed\worldbank_selected_indicators_clean.csv


### Create separate clean files

In [7]:
population_clean = wb_clean[
    wb_clean["indicator_code"].isin(["SP.POP.TOTL", "SP.POP.GROW"])
].copy()

economy_clean = wb_clean[
    wb_clean["indicator_code"].isin([
        "NY.GDP.MKTP.CD",
        "NY.GDP.MKTP.KD.ZG",
        "NY.GDP.PCAP.CD",
        "BX.TRF.PWKR.DT.GD.ZS",
        "BX.KLT.DINV.WD.GD.ZS"
    ])
].copy()

unemployment_clean = wb_clean[
    wb_clean["indicator_code"] == "SL.UEM.TOTL.ZS"
].copy()

net_migration_clean = wb_clean[
    wb_clean["indicator_code"] == "SM.POP.NETM"
].copy()

population_clean.to_csv(PROCESSED_DIR / "population_clean.csv", index=False)
economy_clean.to_csv(PROCESSED_DIR / "economy_clean.csv", index=False)
unemployment_clean.to_csv(PROCESSED_DIR / "unemployment_clean.csv", index=False)
net_migration_clean.to_csv(PROCESSED_DIR / "net_migration_clean.csv", index=False)

print("Saved population_clean.csv")
print("Saved economy_clean.csv")
print("Saved unemployment_clean.csv")
print("Saved net_migration_clean.csv")

Saved population_clean.csv
Saved economy_clean.csv
Saved unemployment_clean.csv
Saved net_migration_clean.csv


#### Check if files were created

In [8]:
processed_files = list(PROCESSED_DIR.glob("*.csv"))

for file in processed_files:
    print(file.name)

economy_clean.csv
net_migration_clean.csv
population_clean.csv
unemployment_clean.csv
worldbank_albania_clean.csv
worldbank_selected_indicators_clean.csv


# Eurostat Data Cleaning

In this section, we clean Eurostat migration and residence permit datasets.

The Eurostat datasets are used to study:

- Albanian citizens immigrating into European reporting countries
- Albanian citizens emigrating from European reporting countries
- First residence permits issued to Albanian citizens
- Residence permits by reason, age, and sex

Important data rules:

1. Immigration into European countries does not directly mean emigration from Albania.
2. EU aggregate rows such as EU27_2020 and EU28 must not be mixed with individual country rows.
3. TOTAL categories must not be summed together with subcategories.
4. Residence permit categories must be analyzed carefully to avoid double counting.

In [10]:
EUROSTAT_DIR = RAW_DIR / "eurostat"

immigration_path = EUROSTAT_DIR / "eurostat_albanian_citizens_immigration.csv.gz"
emigration_path = EUROSTAT_DIR / "eurostat_albanian_citizens_emigration.csv.gz"
first_permits_path = EUROSTAT_DIR / "eurostat_first_residence_permits_albania.csv.gz"
permits_age_sex_reason_path = EUROSTAT_DIR / "eurostat_residence_permits_age_sex_reason_albania.csv.gz"

print("Eurostat folder:", EUROSTAT_DIR)
print("Immigration file exists:", immigration_path.exists())
print("Emigration file exists:", emigration_path.exists())
print("First permits file exists:", first_permits_path.exists())
print("Age/Sex/Reason permits file exists:", permits_age_sex_reason_path.exists())

Eurostat folder: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\data\raw\eurostat
Immigration file exists: True
Emigration file exists: True
First permits file exists: True
Age/Sex/Reason permits file exists: True


## Clean Migration Flow Data

In [11]:
def clean_migration_flow_file(file_path, flow_type):
    """
    Cleans Eurostat migration flow files.
    
    flow_type should be:
    - immigration
    - emigration
    """
    
    df = pd.read_csv(file_path)
    
    clean = df[
        [
            "citizen",
            "Country of citizenship",
            "geo",
            "Geopolitical entity (reporting)",
            "TIME_PERIOD",
            "OBS_VALUE"
        ]
    ].copy()
    
    clean = clean.rename(columns={
        "citizen": "citizenship_code",
        "Country of citizenship": "citizenship_country",
        "geo": "reporting_country_code",
        "Geopolitical entity (reporting)": "reporting_country",
        "TIME_PERIOD": "year",
        "OBS_VALUE": "value"
    })
    
    clean["flow_type"] = flow_type
    clean["year"] = pd.to_numeric(clean["year"], errors="coerce").astype("Int64")
    clean["value"] = pd.to_numeric(clean["value"], errors="coerce")
    
    clean = clean.dropna(subset=["year", "value"])
    clean = clean[clean["value"] >= 0]
    
    aggregate_geos = ["EU27_2020", "EU28"]
    clean["is_aggregate_reporting_geo"] = clean["reporting_country_code"].isin(aggregate_geos)
    
    clean = clean[
        [
            "flow_type",
            "citizenship_code",
            "citizenship_country",
            "reporting_country_code",
            "reporting_country",
            "year",
            "value",
            "is_aggregate_reporting_geo"
        ]
    ]
    
    return clean


immigration_clean = clean_migration_flow_file(immigration_path, "immigration")
emigration_clean = clean_migration_flow_file(emigration_path, "emigration")

migration_flows_clean = pd.concat(
    [immigration_clean, emigration_clean],
    ignore_index=True
)

migration_flows_clean.head()

,flow_type,citizenship_code,citizenship_country,reporting_country_code,reporting_country,year,value,is_aggregate_reporting_geo
0,immigration,AL,Albania,AT,Austria,2015,335.0,False
1,immigration,AL,Albania,AT,Austria,2016,337.0,False
2,immigration,AL,Albania,AT,Austria,2017,381.0,False
3,immigration,AL,Albania,AT,Austria,2018,364.0,False
4,immigration,AL,Albania,AT,Austria,2019,410.0,False


## Check Migration Data

In [12]:
print("Migration flow rows:", migration_flows_clean.shape[0])
print("Years:", migration_flows_clean["year"].min(), "-", migration_flows_clean["year"].max())
print("Flow types:", migration_flows_clean["flow_type"].unique())

migration_flows_clean.groupby("flow_type")["value"].describe()

Migration flow rows: 3955
Years: 2015 - 2024
Flow types: <ArrowStringArray>
['immigration', 'emigration']
Length: 2, dtype: str


,count,mean,std,min,25%,50%,75%,max
flow_type,,,,,,,,
emigration,1959.0,50290.692700,220701.820501,0.0,8.0,1257.0,24284.50,3166449.0
immigration,1996.0,76175.719439,374096.052549,0.0,26.0,2158.5,30519.25,6997284.0


## Save Migration Flow Files

In [13]:
migration_flows_clean.to_csv(
    PROCESSED_DIR / "migration_flows_clean.csv",
    index=False
)

migration_flows_country_clean = migration_flows_clean[
    migration_flows_clean["is_aggregate_reporting_geo"] == False
].copy()

migration_flows_country_clean.to_csv(
    PROCESSED_DIR / "migration_flows_country_clean.csv",
    index=False
)

print("Saved migration_flows_clean.csv")
print("Saved migration_flows_country_clean.csv")

Saved migration_flows_clean.csv
Saved migration_flows_country_clean.csv


## Preview Top Immigration Destinations

In [14]:
top_immigration_destinations = (
    migration_flows_country_clean[
        migration_flows_country_clean["flow_type"] == "immigration"
    ]
    .groupby("reporting_country", as_index=False)["value"]
    .sum()
    .sort_values("value", ascending=False)
)

top_immigration_destinations.head(10)

,reporting_country,value
10,Germany,16828903.0
31,Spain,11146851.0
9,France,6548079.0
34,United Kingdom,6299026.0
15,Italy,5578003.0
26,Poland,3833529.0
28,Romania,3703357.0
23,Netherlands,3381532.0
33,Switzerland,2514905.0
1,Belgium,2261534.0


## Cleaning First Residence Permits

This dataset shows first residence permits issued to Albanian citizens by European reporting countries.

The main categories are:

- TOTAL
- Employment
- Education
- Family

Important: TOTAL should not be added together with Employment, Education, and Family because that causes double counting.

In [15]:
first_permits_raw = pd.read_csv(first_permits_path)

first_permits_clean = first_permits_raw[
    [
        "reason",
        "Reason",
        "citizen",
        "Country of citizenship",
        "duration",
        "Duration",
        "geo",
        "Geopolitical entity (reporting)",
        "TIME_PERIOD",
        "OBS_VALUE"
    ]
].copy()

first_permits_clean = first_permits_clean.rename(columns={
    "reason": "reason_code",
    "Reason": "reason",
    "citizen": "citizenship_code",
    "Country of citizenship": "citizenship_country",
    "duration": "duration_code",
    "Duration": "duration",
    "geo": "reporting_country_code",
    "Geopolitical entity (reporting)": "reporting_country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "value"
})

first_permits_clean["year"] = pd.to_numeric(first_permits_clean["year"], errors="coerce").astype("Int64")
first_permits_clean["value"] = pd.to_numeric(first_permits_clean["value"], errors="coerce")

first_permits_clean = first_permits_clean.dropna(subset=["year", "value"])
first_permits_clean = first_permits_clean[first_permits_clean["value"] >= 0]

aggregate_geos = ["EU27_2020", "EU28"]
first_permits_clean["is_aggregate_reporting_geo"] = first_permits_clean["reporting_country_code"].isin(aggregate_geos)
first_permits_clean["is_total_reason"] = first_permits_clean["reason_code"] == "TOTAL"

first_permits_clean.head()

,reason_code,reason,citizenship_code,citizenship_country,duration_code,duration,reporting_country_code,reporting_country,year,value,is_aggregate_reporting_geo,is_total_reason
0,EDUC,Education reasons,AL,Albania,TOTAL,Total,AT,Austria,2008,39,False,False
1,EDUC,Education reasons,AL,Albania,TOTAL,Total,AT,Austria,2009,48,False,False
2,EDUC,Education reasons,AL,Albania,TOTAL,Total,AT,Austria,2010,48,False,False
3,EDUC,Education reasons,AL,Albania,TOTAL,Total,AT,Austria,2011,69,False,False
4,EDUC,Education reasons,AL,Albania,TOTAL,Total,AT,Austria,2012,86,False,False


## Save Residence Permit File

In [16]:
first_permits_clean.to_csv(
    PROCESSED_DIR / "residence_permits_clean.csv",
    index=False
)

residence_permits_country_clean = first_permits_clean[
    first_permits_clean["is_aggregate_reporting_geo"] == False
].copy()

residence_permits_country_clean.to_csv(
    PROCESSED_DIR / "residence_permits_country_clean.csv",
    index=False
)

residence_permits_total_clean = residence_permits_country_clean[
    residence_permits_country_clean["reason_code"] == "TOTAL"
].copy()

residence_permits_reason_breakdown_clean = residence_permits_country_clean[
    residence_permits_country_clean["reason_code"] != "TOTAL"
].copy()

residence_permits_total_clean.to_csv(
    PROCESSED_DIR / "residence_permits_total_clean.csv",
    index=False
)

residence_permits_reason_breakdown_clean.to_csv(
    PROCESSED_DIR / "residence_permits_reason_breakdown_clean.csv",
    index=False
)

print("Saved residence_permits_clean.csv")
print("Saved residence_permits_country_clean.csv")
print("Saved residence_permits_total_clean.csv")
print("Saved residence_permits_reason_breakdown_clean.csv")

Saved residence_permits_clean.csv
Saved residence_permits_country_clean.csv
Saved residence_permits_total_clean.csv
Saved residence_permits_reason_breakdown_clean.csv


## Check Residence Permit Categories

In [17]:
residence_permits_country_clean["reason"].value_counts()

reason
Education reasons     540
Employment reasons    540
Family reasons        540
Total                 536
Name: count, dtype: int64

In [18]:
permits_by_reason_preview = (
    residence_permits_reason_breakdown_clean
    .groupby(["year", "reason"], as_index=False)["value"]
    .sum()
    .sort_values(["year", "reason"])
)

permits_by_reason_preview.head(20)

,year,reason,value
0,2008,Education reasons,407006
1,2008,Employment reasons,737366
2,2008,Family reasons,681431
3,2009,Education reasons,470809
4,2009,Employment reasons,618918
5,2009,Family reasons,668135
6,2010,Education reasons,477621
7,2010,Employment reasons,778042
8,2010,Family reasons,779178
9,2011,Education reasons,457699


## Clean Age/Sex/Reason Residence Permits

This dataset is more detailed. It contains residence permits by:

- Reason
- Sex
- Age group
- Reporting country
- Year

We keep flags for TOTAL age, TOTAL sex, and TOTAL reason so we can avoid double counting later.

In [19]:
permits_age_raw = pd.read_csv(permits_age_sex_reason_path)

permits_age_clean = permits_age_raw[
    [
        "reason",
        "Reason",
        "sex",
        "Sex",
        "age",
        "Age class",
        "citizen",
        "Country of citizenship",
        "geo",
        "Geopolitical entity (reporting)",
        "TIME_PERIOD",
        "OBS_VALUE"
    ]
].copy()

permits_age_clean = permits_age_clean.rename(columns={
    "reason": "reason_code",
    "Reason": "reason",
    "sex": "sex_code",
    "Sex": "sex",
    "age": "age_code",
    "Age class": "age_group",
    "citizen": "citizenship_code",
    "Country of citizenship": "citizenship_country",
    "geo": "reporting_country_code",
    "Geopolitical entity (reporting)": "reporting_country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "value"
})

permits_age_clean["year"] = pd.to_numeric(permits_age_clean["year"], errors="coerce").astype("Int64")
permits_age_clean["value"] = pd.to_numeric(permits_age_clean["value"], errors="coerce")

permits_age_clean = permits_age_clean.dropna(subset=["year", "value"])
permits_age_clean = permits_age_clean[permits_age_clean["value"] >= 0]

permits_age_clean["is_aggregate_reporting_geo"] = permits_age_clean["reporting_country_code"].isin(["EU27_2020", "EU28"])
permits_age_clean["is_total_reason"] = permits_age_clean["reason_code"] == "TOTAL"
permits_age_clean["is_total_sex"] = permits_age_clean["sex_code"] == "T"
permits_age_clean["is_total_age"] = permits_age_clean["age_code"] == "TOTAL"
permits_age_clean["is_unknown_sex"] = permits_age_clean["sex_code"] == "UNK"

permits_age_clean.head()

,reason_code,reason,sex_code,sex,age_code,age_group,citizenship_code,citizenship_country,reporting_country_code,reporting_country,year,value,is_aggregate_reporting_geo,is_total_reason,is_total_sex,is_total_age,is_unknown_sex
0,EDUC,Education reasons,F,Females,TOTAL,Total,AL,Albania,AT,Austria,2011,36.0,False,False,False,True,False
1,EDUC,Education reasons,F,Females,TOTAL,Total,AL,Albania,AT,Austria,2012,42.0,False,False,False,True,False
2,EDUC,Education reasons,F,Females,TOTAL,Total,AL,Albania,AT,Austria,2013,41.0,False,False,False,True,False
3,EDUC,Education reasons,F,Females,TOTAL,Total,AL,Albania,AT,Austria,2014,44.0,False,False,False,True,False
4,EDUC,Education reasons,F,Females,TOTAL,Total,AL,Albania,AT,Austria,2015,75.0,False,False,False,True,False


## Save Age/Sex/Reason File

In [20]:
permits_age_clean.to_csv(
    PROCESSED_DIR / "residence_permits_age_sex_reason_clean.csv",
    index=False
)

permits_age_country_clean = permits_age_clean[
    permits_age_clean["is_aggregate_reporting_geo"] == False
].copy()

permits_age_country_clean.to_csv(
    PROCESSED_DIR / "residence_permits_age_sex_reason_country_clean.csv",
    index=False
)

print("Saved residence_permits_age_sex_reason_clean.csv")
print("Saved residence_permits_age_sex_reason_country_clean.csv")

Saved residence_permits_age_sex_reason_clean.csv
Saved residence_permits_age_sex_reason_country_clean.csv


In [21]:
processed_files = sorted(PROCESSED_DIR.glob("*.csv"))

for file in processed_files:
    print(file.name)

economy_clean.csv
migration_flows_clean.csv
migration_flows_country_clean.csv
net_migration_clean.csv
population_clean.csv
residence_permits_age_sex_reason_clean.csv
residence_permits_age_sex_reason_country_clean.csv
residence_permits_clean.csv
residence_permits_country_clean.csv
residence_permits_reason_breakdown_clean.csv
residence_permits_total_clean.csv
unemployment_clean.csv
worldbank_albania_clean.csv
worldbank_selected_indicators_clean.csv


# Master Analysis Dataset

In this section, we combine the cleaned datasets into one year-based dataset.

The master dataset will include:

- Population total
- Population growth
- Net migration
- GDP
- GDP growth
- GDP per capita
- Unemployment
- Remittances
- FDI
- Eurostat immigration flow
- Eurostat emigration flow
- Total first residence permits
- Residence permits by reason
- Permit reason shares

This dataset will be used for statistical analysis, forecasting, and Power BI.

## Load Cleaned Files

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT_DIR = Path.cwd().parent
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

worldbank_clean = pd.read_csv(PROCESSED_DIR / "worldbank_selected_indicators_clean.csv")
migration_flows_country = pd.read_csv(PROCESSED_DIR / "migration_flows_country_clean.csv")
residence_permits_total = pd.read_csv(PROCESSED_DIR / "residence_permits_total_clean.csv")
residence_permits_reason = pd.read_csv(PROCESSED_DIR / "residence_permits_reason_breakdown_clean.csv")

print("World Bank:", worldbank_clean.shape)
print("Migration flows:", migration_flows_country.shape)
print("Residence permits total:", residence_permits_total.shape)
print("Residence permits reason:", residence_permits_reason.shape)

World Bank: (227, 7)
Migration flows: (3859, 8)
Residence permits total: (536, 12)
Residence permits reason: (1620, 12)


## Convert World Bank Data to Wide Format

In [23]:
worldbank_wide = (
    worldbank_clean
    .pivot_table(
        index="year",
        columns="indicator_short_name",
        values="value",
        aggfunc="first"
    )
    .reset_index()
)

worldbank_wide.columns.name = None

worldbank_wide.head()

,year,fdi_percent_gdp,gdp_current_usd,gdp_growth_annual_percent,gdp_per_capita_current_usd,net_migration,population_growth_annual_percent,population_total,remittances_percent_gdp,unemployment_total_percent
0,2000,3.989321,3.584570e+09,7.462859,1160.420471,-60531.0,-0.637357,3089027.0,16.677034,19.023
1,2001,5.107089,4.059064e+09,8.863731,1326.416524,-48070.0,-0.938470,3060173.0,17.228110,18.570
2,2002,2.990031,4.515003e+09,4.628396,1479.838846,-45178.0,-0.299877,3051010.0,16.247386,17.891
3,2003,3.068687,5.801712e+09,5.333264,1908.699007,-48517.0,-0.374149,3039616.0,15.318730,16.985
4,2004,4.607823,7.406646e+09,5.266262,2446.909499,-48654.0,-0.417931,3026939.0,15.670685,16.306


## Create Migration Flow Yearly Summary

In [24]:
migration_yearly = (
    migration_flows_country
    .groupby(["year", "flow_type"], as_index=False)["value"]
    .sum()
)

migration_yearly_wide = (
    migration_yearly
    .pivot_table(
        index="year",
        columns="flow_type",
        values="value",
        aggfunc="sum"
    )
    .reset_index()
)

migration_yearly_wide.columns.name = None

migration_yearly_wide = migration_yearly_wide.rename(columns={
    "immigration": "eurostat_immigration_to_europe",
    "emigration": "eurostat_emigration_from_europe"
})

migration_yearly_wide.head()

,year,eurostat_emigration_from_europe,eurostat_immigration_to_europe
0,2015,6078939.0,10169201.0
1,2016,6625508.0,9341215.0
2,2017,6565859.0,9501195.0
3,2018,6169837.0,9780093.0
4,2019,6517560.0,10483989.0


## Create Residence Permits Yearly Summary

In [25]:
permits_total_yearly = (
    residence_permits_total
    .groupby("year", as_index=False)["value"]
    .sum()
    .rename(columns={
        "value": "total_first_residence_permits"
    })
)

permits_total_yearly.head()

,year,total_first_residence_permits
0,2008,2424872
1,2009,2279189
2,2010,2417235
3,2011,2084134
4,2012,2037669


## Create Residence Permits by Reason

In [26]:
permits_reason_yearly = (
    residence_permits_reason
    .groupby(["year", "reason_code"], as_index=False)["value"]
    .sum()
)

permits_reason_wide = (
    permits_reason_yearly
    .pivot_table(
        index="year",
        columns="reason_code",
        values="value",
        aggfunc="sum"
    )
    .reset_index()
)

permits_reason_wide.columns.name = None

permits_reason_wide.head()

,year,EDUC,EMP,FAM
0,2008,407006,737366,681431
1,2009,470809,618918,668135
2,2010,477621,778042,779178
3,2011,457699,491321,710126
4,2012,426266,456380,678707


## Rename Permit Reason Columns

In [27]:
permits_reason_wide = permits_reason_wide.rename(columns={
    "EDUC": "education_permits",
    "EMP": "employment_permits",
    "FAM": "family_permits",
    "OTHER": "other_reason_permits",
    "OTH": "other_reason_permits"
})

permits_reason_wide.head()

,year,education_permits,employment_permits,family_permits
0,2008,407006,737366,681431
1,2009,470809,618918,668135
2,2010,477621,778042,779178
3,2011,457699,491321,710126
4,2012,426266,456380,678707


## Merge Everything Together

In [28]:
master = worldbank_wide.merge(
    migration_yearly_wide,
    on="year",
    how="outer"
)

master = master.merge(
    permits_total_yearly,
    on="year",
    how="outer"
)

master = master.merge(
    permits_reason_wide,
    on="year",
    how="outer"
)

master = master.sort_values("year").reset_index(drop=True)

master.head()

,year,fdi_percent_gdp,gdp_current_usd,gdp_growth_annual_percent,gdp_per_capita_current_usd,net_migration,population_growth_annual_percent,population_total,remittances_percent_gdp,unemployment_total_percent,eurostat_emigration_from_europe,eurostat_immigration_to_europe,total_first_residence_permits,education_permits,employment_permits,family_permits
0,2000,3.989321,3.584570e+09,7.462859,1160.420471,-60531.0,-0.637357,3089027.0,16.677034,19.023,NaN,NaN,NaN,NaN,NaN,NaN
1,2001,5.107089,4.059064e+09,8.863731,1326.416524,-48070.0,-0.938470,3060173.0,17.228110,18.570,NaN,NaN,NaN,NaN,NaN,NaN
2,2002,2.990031,4.515003e+09,4.628396,1479.838846,-45178.0,-0.299877,3051010.0,16.247386,17.891,NaN,NaN,NaN,NaN,NaN,NaN
3,2003,3.068687,5.801712e+09,5.333264,1908.699007,-48517.0,-0.374149,3039616.0,15.318730,16.985,NaN,NaN,NaN,NaN,NaN,NaN
4,2004,4.607823,7.406646e+09,5.266262,2446.909499,-48654.0,-0.417931,3026939.0,15.670685,16.306,NaN,NaN,NaN,NaN,NaN,NaN


## Filter Main Analysis Period

In [29]:
master = master[master["year"] >= 2000].copy()

master.head()

,year,fdi_percent_gdp,gdp_current_usd,gdp_growth_annual_percent,gdp_per_capita_current_usd,net_migration,population_growth_annual_percent,population_total,remittances_percent_gdp,unemployment_total_percent,eurostat_emigration_from_europe,eurostat_immigration_to_europe,total_first_residence_permits,education_permits,employment_permits,family_permits
0,2000,3.989321,3.584570e+09,7.462859,1160.420471,-60531.0,-0.637357,3089027.0,16.677034,19.023,NaN,NaN,NaN,NaN,NaN,NaN
1,2001,5.107089,4.059064e+09,8.863731,1326.416524,-48070.0,-0.938470,3060173.0,17.228110,18.570,NaN,NaN,NaN,NaN,NaN,NaN
2,2002,2.990031,4.515003e+09,4.628396,1479.838846,-45178.0,-0.299877,3051010.0,16.247386,17.891,NaN,NaN,NaN,NaN,NaN,NaN
3,2003,3.068687,5.801712e+09,5.333264,1908.699007,-48517.0,-0.374149,3039616.0,15.318730,16.985,NaN,NaN,NaN,NaN,NaN,NaN
4,2004,4.607823,7.406646e+09,5.266262,2446.909499,-48654.0,-0.417931,3026939.0,15.670685,16.306,NaN,NaN,NaN,NaN,NaN,NaN


## Create Feature Engineering Columns

In [31]:
# Migration rate per 1,000 population
master["net_migration_rate_per_1000"] = (
    master["net_migration"] / master["population_total"] * 1000
)

# Residence permits per 100,000 population
master["residence_permits_per_100k_population"] = (
    master["total_first_residence_permits"] / master["population_total"] * 100000
)

# Permit reason shares
master["employment_permit_share"] = (
    master["employment_permits"] / master["total_first_residence_permits"]
)

master["education_permit_share"] = (
    master["education_permits"] / master["total_first_residence_permits"]
)

master["family_permit_share"] = (
    master["family_permits"] / master["total_first_residence_permits"]
)

# Migration pressure index
master["migration_pressure_index"] = (
    master["net_migration_rate_per_1000"].abs()
    + master["residence_permits_per_100k_population"].fillna(0)
)

master.head()

,year,fdi_percent_gdp,gdp_current_usd,gdp_growth_annual_percent,gdp_per_capita_current_usd,net_migration,population_growth_annual_percent,population_total,remittances_percent_gdp,unemployment_total_percent,...,total_first_residence_permits,education_permits,employment_permits,family_permits,net_migration_rate_per_1000,residence_permits_per_100k_population,employment_permit_share,education_permit_share,family_permit_share,migration_pressure_index
0,2000,3.989321,3.584570e+09,7.462859,1160.420471,-60531.0,-0.637357,3089027.0,16.677034,19.023,...,NaN,NaN,NaN,NaN,-19.595491,NaN,NaN,NaN,NaN,19.595491
1,2001,5.107089,4.059064e+09,8.863731,1326.416524,-48070.0,-0.938470,3060173.0,17.228110,18.570,...,NaN,NaN,NaN,NaN,-15.708262,NaN,NaN,NaN,NaN,15.708262
2,2002,2.990031,4.515003e+09,4.628396,1479.838846,-45178.0,-0.299877,3051010.0,16.247386,17.891,...,NaN,NaN,NaN,NaN,-14.807556,NaN,NaN,NaN,NaN,14.807556
3,2003,3.068687,5.801712e+09,5.333264,1908.699007,-48517.0,-0.374149,3039616.0,15.318730,16.985,...,NaN,NaN,NaN,NaN,-15.961556,NaN,NaN,NaN,NaN,15.961556
4,2004,4.607823,7.406646e+09,5.266262,2446.909499,-48654.0,-0.417931,3026939.0,15.670685,16.306,...,NaN,NaN,NaN,NaN,-16.073664,NaN,NaN,NaN,NaN,16.073664


## Check Missing Values

In [32]:
missing_summary = (
    master
    .isna()
    .sum()
    .reset_index()
)

missing_summary.columns = ["column", "missing_values"]

missing_summary.sort_values("missing_values", ascending=False)

,column,missing_values
10,eurostat_emigration_from_europe,16
11,eurostat_immigration_to_europe,16
17,residence_permits_per_100k_population,9
13,education_permits,8
15,family_permits,8
18,employment_permit_share,8
20,family_permit_share,8
19,education_permit_share,8
14,employment_permits,8
12,total_first_residence_permits,8


## Save the Master Dataset

In [49]:
master_output_path = PROCESSED_DIR / "master_analysis_dataset.csv"
# Ensure required feature columns exist before saving

if "net_migration_rate_per_1000" not in master.columns:
    master["net_migration_rate_per_1000"] = (
        master["net_migration"] / master["population_total"] * 1000
    )

if "residence_permits_per_100k_population" not in master.columns:
    master["residence_permits_per_100k_population"] = (
        master["total_first_residence_permits"] / master["population_total"] * 100000
    )

if "employment_permit_share" not in master.columns:
    master["employment_permit_share"] = (
        master["employment_permits"] / master["total_first_residence_permits"]
    )

if "education_permit_share" not in master.columns:
    master["education_permit_share"] = (
        master["education_permits"] / master["total_first_residence_permits"]
    )

if "family_permit_share" not in master.columns:
    master["family_permit_share"] = (
        master["family_permits"] / master["total_first_residence_permits"]
    )

if "migration_pressure_index" not in master.columns:
    master["migration_pressure_index"] = (
        master["net_migration_rate_per_1000"].abs()
        + master["residence_permits_per_100k_population"].fillna(0)
    )
master.to_csv(master_output_path, index=False)

print("Saved:", master_output_path)
print("Rows:", master.shape[0])
print("Columns:", master.shape[1])

Saved: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\data\processed\master_analysis_dataset.csv
Rows: 26
Columns: 22


In [50]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent
PROCESSED_DIR = ROOT_DIR / "data" / "processed"

master_path = PROCESSED_DIR / "master_analysis_dataset.csv"

master = pd.read_csv(master_path)

print("Original columns:")
print(master.columns.tolist())

expected_sql_columns = [
    "year",
    "population_total",
    "population_growth_annual_percent",
    "net_migration",
    "gdp_current_usd",
    "gdp_growth_annual_percent",
    "gdp_per_capita_current_usd",
    "unemployment_total_percent",
    "remittances_percent_gdp",
    "fdi_percent_gdp",
    "eurostat_immigration_to_europe",
    "eurostat_emigration_from_europe",
    "total_first_residence_permits",
    "education_permits",
    "employment_permits",
    "family_permits",
    "other_reason_permits",
    "net_migration_rate_per_1000",
    "residence_permits_per_100k_population",
    "employment_permit_share",
    "education_permit_share",
    "family_permit_share",
    "migration_pressure_index"
]

# Create missing columns if any are not present
for col in expected_sql_columns:
    if col not in master.columns:
        master[col] = np.nan
        print("Created missing column:", col)

# Reorder columns exactly like PostgreSQL staging table
master_sql_ready = master[expected_sql_columns].copy()

sql_ready_path = PROCESSED_DIR / "master_analysis_dataset_sql_ready.csv"

master_sql_ready.to_csv(sql_ready_path, index=False)

print("Saved SQL-ready file:", sql_ready_path)
print("Rows:", master_sql_ready.shape[0])
print("Columns:", master_sql_ready.shape[1])

master_sql_ready.head()

Original columns:
['year', 'fdi_percent_gdp', 'gdp_current_usd', 'gdp_growth_annual_percent', 'gdp_per_capita_current_usd', 'net_migration', 'population_growth_annual_percent', 'population_total', 'remittances_percent_gdp', 'unemployment_total_percent', 'eurostat_emigration_from_europe', 'eurostat_immigration_to_europe', 'total_first_residence_permits', 'education_permits', 'employment_permits', 'family_permits', 'net_migration_rate_per_1000', 'residence_permits_per_100k_population', 'employment_permit_share', 'education_permit_share', 'family_permit_share', 'migration_pressure_index']
Created missing column: other_reason_permits
Saved SQL-ready file: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\data\processed\master_analysis_dataset_sql_ready.csv
Rows: 26
Columns: 23


,year,population_total,population_growth_annual_percent,net_migration,gdp_current_usd,gdp_growth_annual_percent,gdp_per_capita_current_usd,unemployment_total_percent,remittances_percent_gdp,fdi_percent_gdp,...,education_permits,employment_permits,family_permits,other_reason_permits,net_migration_rate_per_1000,residence_permits_per_100k_population,employment_permit_share,education_permit_share,family_permit_share,migration_pressure_index
0,2000,3089027.0,-0.637357,-60531.0,3.584570e+09,7.462859,1160.420471,19.023,16.677034,3.989321,...,NaN,NaN,NaN,NaN,-19.595491,NaN,NaN,NaN,NaN,19.595491
1,2001,3060173.0,-0.938470,-48070.0,4.059064e+09,8.863731,1326.416524,18.570,17.228110,5.107089,...,NaN,NaN,NaN,NaN,-15.708262,NaN,NaN,NaN,NaN,15.708262
2,2002,3051010.0,-0.299877,-45178.0,4.515003e+09,4.628396,1479.838846,17.891,16.247386,2.990031,...,NaN,NaN,NaN,NaN,-14.807556,NaN,NaN,NaN,NaN,14.807556
3,2003,3039616.0,-0.374149,-48517.0,5.801712e+09,5.333264,1908.699007,16.985,15.318730,3.068687,...,NaN,NaN,NaN,NaN,-15.961556,NaN,NaN,NaN,NaN,15.961556
4,2004,3026939.0,-0.417931,-48654.0,7.406646e+09,5.266262,2446.909499,16.306,15.670685,4.607823,...,NaN,NaN,NaN,NaN,-16.073664,NaN,NaN,NaN,NaN,16.073664


## Preview the Final Master Dataset

In [34]:
master.tail(10)

,year,fdi_percent_gdp,gdp_current_usd,gdp_growth_annual_percent,gdp_per_capita_current_usd,net_migration,population_growth_annual_percent,population_total,remittances_percent_gdp,unemployment_total_percent,...,total_first_residence_permits,education_permits,employment_permits,family_permits,net_migration_rate_per_1000,residence_permits_per_100k_population,employment_permit_share,education_permit_share,family_permit_share,migration_pressure_index
16,2016,8.711472,1.198867e+10,3.908936,4457.634122,-9347.0,-1.543135,2689469.0,10.893696,15.418,...,2911426.0,451813.0,809781.0,780791.0,-3.475407,108252.818679,0.278139,0.155186,0.268182,108256.294086
17,2017,7.714113,1.325827e+10,3.283176,5006.360130,-14901.0,-1.543152,2648285.0,9.894372,13.616,...,3064248.0,475071.0,936643.0,833617.0,-5.626660,115706.882001,0.305668,0.155037,0.272046,115712.508661
18,2018,7.831091,1.537951e+10,3.671419,5897.654526,-15032.0,-1.543100,2607733.0,9.481513,12.304,...,3047171.0,525927.0,979403.0,913950.0,-5.764394,116851.341759,0.321414,0.172595,0.299934,116857.106153
19,2019,7.706215,1.558511e+10,2.062578,6069.439031,-23094.0,-1.543137,2567801.0,9.450123,11.466,...,2695223.0,336665.0,1048369.0,793904.0,-8.993688,104962.300427,0.388973,0.124912,0.294560,104971.294115
20,2020,7.018652,1.524146e+10,-3.313756,6027.913507,-16680.0,-1.543156,2528480.0,9.618418,11.639,...,2124254.0,248521.0,834431.0,611169.0,-6.596849,84013.082959,0.392811,0.116992,0.287710,84019.679808
21,2021,6.757915,1.803199e+10,8.969576,7242.455131,-32848.0,-1.543121,2489762.0,9.529486,11.474,...,3079897.0,359982.0,1231662.0,815685.0,-13.193229,123702.466340,0.399904,0.116881,0.264842,123715.659569
22,2022,7.579340,1.901725e+10,4.826801,7756.961887,-16426.0,-1.543157,2451636.0,9.177170,10.785,...,3239522.0,394498.0,1075298.0,912167.0,-6.700016,132137.152497,0.331931,0.121777,0.281575,132143.852513
23,2023,6.900370,2.349124e+10,4.015417,9730.869219,-25357.0,-1.543108,2414095.0,8.667780,10.669,...,3442610.0,455052.0,1049901.0,992164.0,-10.503729,142604.578527,0.304972,0.132182,0.288201,142615.082256
24,2024,6.323495,2.704643e+10,4.045928,11377.775743,-24472.0,-1.543144,2377128.0,8.409393,10.689,...,3141954.0,450113.0,920184.0,922915.0,-10.294776,132174.371763,0.292870,0.143259,0.293739,132184.666539
25,2025,NaN,NaN,NaN,NaN,-24230.0,NaN,NaN,NaN,10.926,...,74903.0,11102.0,7584.0,35478.0,NaN,NaN,0.101251,0.148218,0.473653,NaN


In [35]:
master.info()

<class 'pandas.DataFrame'>
RangeIndex: 26 entries, 0 to 25
Data columns (total 22 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   year                                   26 non-null     int64  
 1   fdi_percent_gdp                        25 non-null     float64
 2   gdp_current_usd                        25 non-null     float64
 3   gdp_growth_annual_percent              25 non-null     float64
 4   gdp_per_capita_current_usd             25 non-null     float64
 5   net_migration                          26 non-null     float64
 6   population_growth_annual_percent       25 non-null     float64
 7   population_total                       25 non-null     float64
 8   remittances_percent_gdp                25 non-null     float64
 9   unemployment_total_percent             26 non-null     float64
 10  eurostat_emigration_from_europe        10 non-null     float64
 11  eurostat_immigratio

##### The master analysis dataset was created by merging World Bank economic and demographic indicators with Eurostat migration and residence permit data by year. World Bank data was used for Albania-level indicators such as population, GDP, unemployment and net migration. Eurostat data was used as a proxy for Albanian citizens' mobility into European reporting countries and for first residence permits issued to Albanian citizens. EU aggregate rows were excluded from country-level calculations, and TOTAL residence permit categories were separated from reason-based subcategories to avoid double counting.

# Data Quality Audit

Before starting the exploratory data analysis, we perform a data quality audit.

This audit checks:

- Missing values
- Duplicate rows
- Year coverage
- Invalid years
- Invalid numeric values
- Outliers
- Dataset limitations
- Source reliability notes

The final result will be saved as:

`reports/data_quality_report.md`

## Load All Processed Files

In [36]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
REPORTS_DIR = ROOT_DIR / "reports"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

processed_files = sorted(PROCESSED_DIR.glob("*.csv"))

print("Processed files found:")

for file in processed_files:
    print("-", file.name)

Processed files found:
- economy_clean.csv
- master_analysis_dataset.csv
- migration_flows_clean.csv
- migration_flows_country_clean.csv
- net_migration_clean.csv
- population_clean.csv
- residence_permits_age_sex_reason_clean.csv
- residence_permits_age_sex_reason_country_clean.csv
- residence_permits_clean.csv
- residence_permits_country_clean.csv
- residence_permits_reason_breakdown_clean.csv
- residence_permits_total_clean.csv
- unemployment_clean.csv
- worldbank_albania_clean.csv
- worldbank_selected_indicators_clean.csv


## Create General Data Quality Summary

In [37]:
data_quality_summary = []

for file in processed_files:
    df = pd.read_csv(file)
    
    year_min = df["year"].min() if "year" in df.columns else None
    year_max = df["year"].max() if "year" in df.columns else None
    
    data_quality_summary.append({
        "file_name": file.name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "duplicate_rows": df.duplicated().sum(),
        "total_missing_values": df.isna().sum().sum(),
        "year_min": year_min,
        "year_max": year_max
    })

data_quality_summary_df = pd.DataFrame(data_quality_summary)

data_quality_summary_df

,file_name,rows,columns,duplicate_rows,total_missing_values,year_min,year_max
0,economy_clean.csv,125,7,0,0,2000,2024
1,master_analysis_dataset.csv,26,22,0,106,2000,2025
2,migration_flows_clean.csv,3955,8,0,0,2015,2024
3,migration_flows_country_clean.csv,3859,8,0,0,2015,2024
4,net_migration_clean.csv,26,7,0,0,2000,2025
5,population_clean.csv,50,7,0,0,2000,2024
6,residence_permits_age_sex_reason_clean.csv,233840,17,0,0,2010,2025
7,residence_permits_age_sex_reason_country_clean...,233160,17,0,0,2010,2025
8,residence_permits_clean.csv,2302,12,0,0,2008,2025
9,residence_permits_country_clean.csv,2156,12,0,0,2008,2025


## Missing Values in Master Dataset

In [38]:
master_path = PROCESSED_DIR / "master_analysis_dataset.csv"

master = pd.read_csv(master_path)

missing_master = (
    master
    .isna()
    .sum()
    .reset_index()
)

missing_master.columns = ["column", "missing_values"]

missing_master["missing_percent"] = (
    missing_master["missing_values"] / len(master) * 100
).round(2)

missing_master = missing_master.sort_values(
    "missing_values", 
    ascending=False
)

missing_master

,column,missing_values,missing_percent
10,eurostat_emigration_from_europe,16,61.54
11,eurostat_immigration_to_europe,16,61.54
17,residence_permits_per_100k_population,9,34.62
13,education_permits,8,30.77
15,family_permits,8,30.77
18,employment_permit_share,8,30.77
20,family_permit_share,8,30.77
19,education_permit_share,8,30.77
14,employment_permits,8,30.77
12,total_first_residence_permits,8,30.77


## Check Duplicate Rows

In [39]:
duplicate_summary = []

for file in processed_files:
    df = pd.read_csv(file)
    
    duplicate_summary.append({
        "file_name": file.name,
        "duplicate_rows": df.duplicated().sum()
    })

duplicate_summary_df = pd.DataFrame(duplicate_summary)

duplicate_summary_df

,file_name,duplicate_rows
0,economy_clean.csv,0
1,master_analysis_dataset.csv,0
2,migration_flows_clean.csv,0
3,migration_flows_country_clean.csv,0
4,net_migration_clean.csv,0
5,population_clean.csv,0
6,residence_permits_age_sex_reason_clean.csv,0
7,residence_permits_age_sex_reason_country_clean...,0
8,residence_permits_clean.csv,0
9,residence_permits_country_clean.csv,0


## Check Invalid Years

In [40]:
invalid_year_summary = []

for file in processed_files:
    df = pd.read_csv(file)
    
    if "year" in df.columns:
        invalid_years = df[
            (df["year"] < 1900) | 
            (df["year"] > 2100) |
            (df["year"].isna())
        ]
        
        invalid_year_summary.append({
            "file_name": file.name,
            "invalid_year_rows": invalid_years.shape[0],
            "min_year": df["year"].min(),
            "max_year": df["year"].max()
        })

invalid_year_summary_df = pd.DataFrame(invalid_year_summary)

invalid_year_summary_df

,file_name,invalid_year_rows,min_year,max_year
0,economy_clean.csv,0,2000,2024
1,master_analysis_dataset.csv,0,2000,2025
2,migration_flows_clean.csv,0,2015,2024
3,migration_flows_country_clean.csv,0,2015,2024
4,net_migration_clean.csv,0,2000,2025
5,population_clean.csv,0,2000,2024
6,residence_permits_age_sex_reason_clean.csv,0,2010,2025
7,residence_permits_age_sex_reason_country_clean...,0,2010,2025
8,residence_permits_clean.csv,0,2008,2025
9,residence_permits_country_clean.csv,0,2008,2025


## Check Invalid Numeric Values

In [41]:
numeric_quality_summary = []

for file in processed_files:
    df = pd.read_csv(file)
    
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    
    for col in numeric_cols:
        numeric_quality_summary.append({
            "file_name": file.name,
            "column": col,
            "missing_values": df[col].isna().sum(),
            "negative_values": (df[col] < 0).sum(),
            "zero_values": (df[col] == 0).sum(),
            "min_value": df[col].min(),
            "max_value": df[col].max()
        })

numeric_quality_summary_df = pd.DataFrame(numeric_quality_summary)

numeric_quality_summary_df.head(30)

,file_name,column,missing_values,negative_values,zero_values,min_value,max_value
0,economy_clean.csv,year,0,0,0,2.000000e+03,2.024000e+03
1,economy_clean.csv,value,0,1,0,-3.313756e+00,2.704643e+10
2,master_analysis_dataset.csv,year,0,0,0,2.000000e+03,2.025000e+03
3,master_analysis_dataset.csv,fdi_percent_gdp,1,0,0,2.990031e+00,1.090682e+01
4,master_analysis_dataset.csv,gdp_current_usd,1,0,0,3.584570e+09,2.704643e+10
5,master_analysis_dataset.csv,gdp_growth_annual_percent,1,1,0,-3.313756e+00,8.969576e+00
6,master_analysis_dataset.csv,gdp_per_capita_current_usd,1,0,0,1.160420e+03,1.137778e+04
7,master_analysis_dataset.csv,net_migration,0,26,0,-6.053100e+04,-9.347000e+03
8,master_analysis_dataset.csv,population_growth_annual_percent,1,25,0,-1.543157e+00,-2.690173e-01
9,master_analysis_dataset.csv,population_total,1,0,0,2.377128e+06,3.089027e+06


## Check Outliers Using IQR

In [42]:
def detect_outliers_iqr(df, column):
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    
    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]
    
    return outliers, lower_bound, upper_bound


outlier_summary = []

important_numeric_columns = [
    "population_total",
    "net_migration",
    "gdp_growth_annual_percent",
    "gdp_per_capita_current_usd",
    "unemployment_total_percent",
    "eurostat_immigration_to_europe",
    "eurostat_emigration_from_europe",
    "total_first_residence_permits",
    "migration_pressure_index"
]

for col in important_numeric_columns:
    if col in master.columns:
        outliers, lower, upper = detect_outliers_iqr(master.dropna(subset=[col]), col)
        
        outlier_summary.append({
            "column": col,
            "outlier_count": outliers.shape[0],
            "lower_bound": round(lower, 2),
            "upper_bound": round(upper, 2)
        })

outlier_summary_df = pd.DataFrame(outlier_summary)

outlier_summary_df

,column,outlier_count,lower_bound,upper_bound
0,population_total,0,2030512.00,3569768.00
1,net_migration,0,-92049.38,23215.62
2,gdp_growth_annual_percent,1,-1.84,9.64
3,gdp_per_capita_current_usd,1,-1202.05,10157.48
4,unemployment_total_percent,0,5.18,22.84
5,eurostat_immigration_to_europe,0,3417707.88,13485964.88
6,eurostat_emigration_from_europe,0,1771646.12,9226019.12
7,total_first_residence_permits,1,992172.50,4300662.50
8,migration_pressure_index,0,-162342.55,270615.60


## Coverage by Year

In [ ]:
coverage_by_year = master.copy()

coverage_by_year["available_indicators"] = coverage_by_year.notna().sum(axis=1)

coverage_by_year_summary = coverage_by_year[
    ["year", "available_indicators"]
].sort_values("year")

coverage_by_year_summary

## Coverage by Indicator

In [44]:
coverage_by_indicator = []

for col in master.columns:
    if col != "year":
        coverage_by_indicator.append({
            "indicator": col,
            "available_years": master[col].notna().sum(),
            "missing_years": master[col].isna().sum(),
            "first_available_year": master.loc[master[col].notna(), "year"].min() if master[col].notna().any() else None,
            "last_available_year": master.loc[master[col].notna(), "year"].max() if master[col].notna().any() else None
        })

coverage_by_indicator_df = pd.DataFrame(coverage_by_indicator)

coverage_by_indicator_df.sort_values("available_years", ascending=False)

,indicator,available_years,missing_years,first_available_year,last_available_year
4,net_migration,26,0,2000,2025
8,unemployment_total_percent,26,0,2000,2025
0,fdi_percent_gdp,25,1,2000,2024
2,gdp_growth_annual_percent,25,1,2000,2024
1,gdp_current_usd,25,1,2000,2024
3,gdp_per_capita_current_usd,25,1,2000,2024
5,population_growth_annual_percent,25,1,2000,2024
6,population_total,25,1,2000,2024
7,remittances_percent_gdp,25,1,2000,2024
20,migration_pressure_index,25,1,2000,2024


## Generate the Data Quality Report

In [47]:
%pip install tabulate

  Using cached tabulate-0.10.0-py3-none-any.whl.metadata (40 kB)
Using cached tabulate-0.10.0-py3-none-any.whl (39 kB)
Note: you may need to restart the kernel to use updated packages.


In [48]:
report_path = REPORTS_DIR / "data_quality_report.md"

report_text = f"""
# Data Quality Report

## Project

**Albania’s Brain Drain: A Data-Driven Analysis of Migration, Employment and Economic Development**

## Purpose of This Report

This report documents the quality of the cleaned datasets used in the analysis.  
The purpose is to identify missing values, duplicate records, year coverage, invalid values, outliers, and limitations before conducting exploratory data analysis, statistical testing, forecasting, and dashboard development.

---

## 1. Processed Dataset Summary

{data_quality_summary_df.to_markdown(index=False)}

---

## 2. Missing Values in Master Dataset

{missing_master.to_markdown(index=False)}

---

## 3. Duplicate Row Summary

{duplicate_summary_df.to_markdown(index=False)}

---

## 4. Invalid Year Summary

{invalid_year_summary_df.to_markdown(index=False)}

---

## 5. Numeric Quality Summary

{numeric_quality_summary_df.to_markdown(index=False)}

---

## 6. Outlier Summary

{outlier_summary_df.to_markdown(index=False)}

---

## 7. Coverage by Year

{coverage_by_year_summary.to_markdown(index=False)}

---

## 8. Coverage by Indicator

{coverage_by_indicator_df.to_markdown(index=False)}

---

## 9. Data Limitations

The datasets used in this project come from reliable public sources, but they have several limitations:

1. World Bank indicators are available for different time periods depending on the indicator.
2. Eurostat immigration data measures Albanian citizens arriving in European reporting countries. It does not directly measure people leaving Albania.
3. Eurostat emigration data measures Albanian citizens leaving European reporting countries. It does not directly measure emigration from Albania.
4. Residence permit data can include TOTAL categories and subcategories. These must be separated to avoid double counting.
5. EU aggregate rows must not be mixed with individual country rows in the same calculation.
6. Some indicators have missing values for recent years.
7. Forecasting results should be interpreted carefully because migration and population trends are affected by policy, economic shocks, labour markets, education, wages, and international conditions.

---

## 10. Source Reliability Notes

The main data sources used in this project are:

- World Bank Open Data
- World Bank Population Estimates
- Eurostat migration and residence permit datasets
- INSTAT census and population reports
- World Bank Albania Migration Survey

These are considered reliable public data sources. However, differences in methodology, reporting coverage, and update frequency may affect comparability across datasets.

---

## 11. Conclusion

The cleaned datasets are suitable for a low-level academic and portfolio-style data analytics project.  
However, the final analysis must clearly explain missing values, proxy indicators, and limitations.  
The project should avoid overclaiming results and should distinguish between direct migration measures and proxy indicators such as Eurostat residence permits and immigration into European countries.
"""

with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_text)

print("Data quality report saved to:", report_path)

Data quality report saved to: C:\Users\mateo\Downloads\albania-brain-drain-analysis-starter(1)\albania-brain-drain-analysis\reports\data_quality_report.md


## Extra tips if any error 
Use this code if not appropiate data is linked to the Eurostat Dataset

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT_DIR = Path.cwd().parent
RAW_DIR = ROOT_DIR / "data" / "raw"
PROCESSED_DIR = ROOT_DIR / "data" / "processed"
POWERBI_DATA_DIR = ROOT_DIR / "dashboard" / "powerbi_data"

EUROSTAT_DIR = RAW_DIR / "eurostat"

immigration_path = EUROSTAT_DIR / "eurostat_albanian_citizens_immigration.csv.gz"
emigration_path = EUROSTAT_DIR / "eurostat_albanian_citizens_emigration.csv.gz"
first_permits_path = EUROSTAT_DIR / "eurostat_first_residence_permits_albania.csv.gz"

POWERBI_DATA_DIR.mkdir(parents=True, exist_ok=True)

def apply_total_filters(df, dataset_type):
    df = df.copy()
    
    # Keep annual data if frequency exists
    if "freq" in df.columns:
        if "A" in df["freq"].dropna().unique():
            df = df[df["freq"] == "A"]
    
    # Keep number/count unit if unit exists
    if "unit" in df.columns:
        if "NR" in df["unit"].dropna().unique():
            df = df[df["unit"] == "NR"]
    
    # Keep total sex only if sex exists
    if "sex" in df.columns:
        if "T" in df["sex"].dropna().unique():
            df = df[df["sex"] == "T"]
    
    # Keep total age only if age exists
    if "age" in df.columns:
        if "TOTAL" in df["age"].dropna().unique():
            df = df[df["age"] == "TOTAL"]
    
    # For residence permits, keep total duration if duration exists
    if dataset_type == "permits":
        if "duration" in df.columns:
            if "TOTAL" in df["duration"].dropna().unique():
                df = df[df["duration"] == "TOTAL"]
    
    return df


def clean_migration_flow(file_path, flow_type):
    raw = pd.read_csv(file_path)
    raw = apply_total_filters(raw, dataset_type="migration")
    
    clean = raw[
        [
            "citizen",
            "Country of citizenship",
            "geo",
            "Geopolitical entity (reporting)",
            "TIME_PERIOD",
            "OBS_VALUE"
        ]
    ].copy()
    
    clean = clean.rename(columns={
        "citizen": "citizenship_code",
        "Country of citizenship": "citizenship_country",
        "geo": "reporting_country_code",
        "Geopolitical entity (reporting)": "reporting_country",
        "TIME_PERIOD": "year",
        "OBS_VALUE": "value"
    })
    
    clean["flow_type"] = flow_type
    clean["year"] = pd.to_numeric(clean["year"], errors="coerce")
    clean["value"] = pd.to_numeric(clean["value"], errors="coerce")
    
    clean = clean.dropna(subset=["year", "value"])
    clean["year"] = clean["year"].astype(int)
    
    aggregate_geos = ["EU27_2020", "EU28", "EA19", "EA20"]
    clean = clean[~clean["reporting_country_code"].isin(aggregate_geos)]
    
    clean = (
        clean
        .groupby(
            [
                "flow_type",
                "citizenship_code",
                "citizenship_country",
                "reporting_country_code",
                "reporting_country",
                "year"
            ],
            as_index=False
        )["value"]
        .sum()
    )
    
    return clean


immigration_clean = clean_migration_flow(immigration_path, "immigration")
emigration_clean = clean_migration_flow(emigration_path, "emigration")

migration_flows_country_clean = pd.concat(
    [immigration_clean, emigration_clean],
    ignore_index=True
)

migration_flows_country_clean.to_csv(
    PROCESSED_DIR / "migration_flows_country_clean.csv",
    index=False
)


# Residence permits
permits_raw = pd.read_csv(first_permits_path)
permits_raw = apply_total_filters(permits_raw, dataset_type="permits")

permits_clean = permits_raw[
    [
        "reason",
        "Reason",
        "citizen",
        "Country of citizenship",
        "duration",
        "Duration",
        "geo",
        "Geopolitical entity (reporting)",
        "TIME_PERIOD",
        "OBS_VALUE"
    ]
].copy()

permits_clean = permits_clean.rename(columns={
    "reason": "reason_code",
    "Reason": "reason",
    "citizen": "citizenship_code",
    "Country of citizenship": "citizenship_country",
    "duration": "duration_code",
    "Duration": "duration",
    "geo": "reporting_country_code",
    "Geopolitical entity (reporting)": "reporting_country",
    "TIME_PERIOD": "year",
    "OBS_VALUE": "value"
})

permits_clean["year"] = pd.to_numeric(permits_clean["year"], errors="coerce")
permits_clean["value"] = pd.to_numeric(permits_clean["value"], errors="coerce")

permits_clean = permits_clean.dropna(subset=["year", "value"])
permits_clean["year"] = permits_clean["year"].astype(int)

aggregate_geos = ["EU27_2020", "EU28", "EA19", "EA20"]
permits_clean = permits_clean[~permits_clean["reporting_country_code"].isin(aggregate_geos)]

permits_clean["is_total_reason"] = permits_clean["reason_code"] == "TOTAL"

residence_permits_country_clean = permits_clean.copy()
residence_permits_total_clean = residence_permits_country_clean[
    residence_permits_country_clean["reason_code"] == "TOTAL"
].copy()

residence_permits_reason_breakdown_clean = residence_permits_country_clean[
    residence_permits_country_clean["reason_code"] != "TOTAL"
].copy()

residence_permits_country_clean.to_csv(
    PROCESSED_DIR / "residence_permits_country_clean.csv",
    index=False
)

residence_permits_total_clean.to_csv(
    PROCESSED_DIR / "residence_permits_total_clean.csv",
    index=False
)

residence_permits_reason_breakdown_clean.to_csv(
    PROCESSED_DIR / "residence_permits_reason_breakdown_clean.csv",
    index=False
)


# Power BI migration yearly file
migration_yearly = (
    migration_flows_country_clean
    .groupby(["year", "flow_type"], as_index=False)["value"]
    .sum()
)

migration_yearly_wide = (
    migration_yearly
    .pivot_table(
        index="year",
        columns="flow_type",
        values="value",
        aggfunc="sum"
    )
    .reset_index()
)

migration_yearly_wide.columns.name = None

migration_yearly_wide = migration_yearly_wide.rename(columns={
    "immigration": "albanian_citizens_immigrating_to_europe",
    "emigration": "albanian_citizens_emigrating_from_europe"
})

migration_yearly_wide.to_csv(
    POWERBI_DATA_DIR / "migration_yearly_dashboard.csv",
    index=False
)


# Power BI destination file
migration_destinations = (
    migration_flows_country_clean[
        migration_flows_country_clean["flow_type"] == "immigration"
    ]
    .groupby(["reporting_country_code", "reporting_country"], as_index=False)["value"]
    .sum()
    .rename(columns={"value": "total_immigration_recorded"})
    .sort_values("total_immigration_recorded", ascending=False)
)

migration_destinations.to_csv(
    POWERBI_DATA_DIR / "migration_destinations_dashboard.csv",
    index=False
)


# Power BI residence permits reason file
residence_reason_dashboard = (
    residence_permits_reason_breakdown_clean
    .groupby(["year", "reason_code", "reason"], as_index=False)["value"]
    .sum()
    .rename(columns={"value": "total_permits"})
)

yearly_total_subcategories = (
    residence_reason_dashboard
    .groupby("year", as_index=False)["total_permits"]
    .sum()
    .rename(columns={"total_permits": "total_reason_subcategory_permits"})
)

residence_reason_dashboard = residence_reason_dashboard.merge(
    yearly_total_subcategories,
    on="year",
    how="left"
)

residence_reason_dashboard["reason_share"] = (
    residence_reason_dashboard["total_permits"] /
    residence_reason_dashboard["total_reason_subcategory_permits"]
)

residence_reason_dashboard.to_csv(
    POWERBI_DATA_DIR / "residence_permits_reason_dashboard.csv",
    index=False
)

print("Corrected Eurostat and Power BI files saved.")

print("\nMigration yearly preview:")
display(migration_yearly_wide.head())

print("\nTop destinations preview:")
display(migration_destinations.head(10))

print("\nResidence permits preview:")
display(residence_reason_dashboard.head())

Corrected Eurostat and Power BI files saved.

Migration yearly preview:


,year,albanian_citizens_emigrating_from_europe,albanian_citizens_immigrating_to_europe
0,2015,6078939.0,10169201.0
1,2016,6625508.0,9341215.0
2,2017,6565859.0,9501195.0
3,2018,6169837.0,9780093.0
4,2019,6517560.0,10483989.0



Top destinations preview:


,reporting_country_code,reporting_country,total_immigration_recorded
6,DE,Germany,16828903.0
10,ES,Spain,11146851.0
12,FR,France,6548079.0
34,UK,United Kingdom,6299026.0
17,IT,Italy,5578003.0
28,PL,Poland,3833529.0
30,RO,Romania,3703357.0
26,NL,Netherlands,3381532.0
3,CH,Switzerland,2514905.0
1,BE,Belgium,2261534.0



Residence permits preview:


,year,reason_code,reason,total_permits,total_reason_subcategory_permits,reason_share
0,2008,EDUC,Education reasons,407006,1825803,0.222919
1,2008,EMP,Employment reasons,737366,1825803,0.403858
2,2008,FAM,Family reasons,681431,1825803,0.373223
3,2009,EDUC,Education reasons,470809,1757862,0.267830
4,2009,EMP,Employment reasons,618918,1757862,0.352086
